# Experiment Runner with Automatic Data Loading

This notebook demonstrates the ExperimentRunner which:
1. **Loads experiment configuration from YAML** - including data loading config
2. **Automatically loads and parses data** - using configured loader and parser
3. **Runs multiple RAG configs** - either sequentially or in parallel
4. **Generates detailed reports** - per-query and aggregate metrics
5. **Compares results** - across different configurations

The experiment configuration in YAML specifies:
- Data source (HuggingFace dataset)
- Data parser (e.g., title_passage)
- RAG config directory
- Number of queries to evaluate
- Parallel execution settings

## 1. Setup & Dependencies

In [1]:
get_ipython().system('pip install datasets faiss-cpu sentence-transformers torch groq python-dotenv nltk pandas -q')

zsh:1: /Users/adityanarayan/aiml-talentsprint/rag-capstone-project/rag-foundry/venv/bin/pip: bad interpreter: /Users/adityanarayan/aiml-talentsprint/rag-capstone-project/real-world-rag-system/venv/bin/python: no such file or directory

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: /Users/adityanarayan/.pyenv/versions/3.11.9/bin/python -m pip install --upgrade pip


## 2. Imports

In [2]:
import os
from dotenv import load_dotenv

from experiment.experiment_config import ExperimentConfig
from experiment.experiment_runner import ExperimentRunner

load_dotenv(override=True)
print("HuggingFace token loaded:", bool(os.getenv("HF_TOKEN")))
print("Groq API key loaded:", bool(os.getenv("GROQ_API_KEY")))

/Users/adityanarayan/aiml-talentsprint/rag-capstone-project/rag-foundry/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HuggingFace token loaded: True
Groq API key loaded: True


## 3. Load Experiment Configuration

The experiment configuration file specifies:
- **data_loader**: How to load data (HuggingFace with dataset_name, subset, split)
- **data_parser**: How to parse documents (title_passage)
- **config_dir**: Directory containing RAG pipeline configs
- **num_queries**: Number of queries to evaluate
- **parallel**: Whether to run configs in parallel

In [3]:
# Load experiment configuration from YAML
EXPERIMENT_CONFIG_PATH = "experiment_configs/covidqa_experiment.yaml"
experiment_config = ExperimentConfig.load(EXPERIMENT_CONFIG_PATH)

print("Experiment Configuration:")
print(f"  Config Dir:  {experiment_config.config_dir}")
print(f"  Report Dir:  {experiment_config.report_dir}")
print(f"  Cache Dir:   {experiment_config.cache_dir}")
print(f"  Num Queries: {experiment_config.num_queries}")
print(f"  Parallel:    {experiment_config.parallel}")
print(f"  Max Workers: {experiment_config.max_workers}")
print(f"\nData Loader:")
print(f"  Type: {experiment_config.data_loader['type']}")
print(f"  Config: {experiment_config.data_loader['config']}")
print(f"\nData Parser:")
print(f"  Type: {experiment_config.data_parser}")

Experiment Configuration:
  Config Dir:  config
  Report Dir:  reports
  Cache Dir:   cache
  Num Queries: 2
  Parallel:    False
  Max Workers: 4

Data Loader:
  Type: huggingface
  Config: {'dataset_name': 'galileo-ai/ragbench', 'subset': 'covidqa', 'split': 'test', 'limit': 100}

Data Parser:
  Type: title_passage


## 4. Initialize Experiment Runner

The ExperimentRunner will:
- Create report directory if it doesn't exist
- Load RAG configs from the specified directory
- Load and parse data automatically based on YAML config

In [4]:
# Initialize experiment runner
runner = ExperimentRunner(experiment_config)
print("ExperimentRunner initialized")

ExperimentRunner initialized


import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

for report in reports:
    print(f"\n{'='*80}")
    print(f"Configuration: {report.config_name}")
    print(f"{'='*80}")
    
    # Display per-query results
    print("\nPer-Query Results:")
    display(report.sections[0].per_query)
    
    # Display aggregate statistics
    print("\nAggregate Statistics:")
    display(report.sections[0].summary)

In [5]:
# Load data automatically based on YAML configuration
print("Loading data based on experiment configuration...")
documents, raw_data = runner.load_data()

print(f"\n✅ Data loaded successfully!")
print(f"  Documents: {len(documents)} parsed documents")
print(f"  Raw Data:  {len(raw_data)} samples")

# Inspect first sample
first_sample = raw_data[0]
print(f"\nFirst Sample:")
print(f"  Question: {first_sample['question'][:100]}...")
print(f"  Documents: {len(first_sample['documents'])}")

Loading data based on experiment configuration...
Loading HuggingFace dataset: galileo-ai/ragbench/covidqa (test)...
Loaded 100 samples

✅ Data loaded successfully!
  Documents: 400 parsed documents
  Raw Data:  100 samples

First Sample:
  Question: Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance?...
  Documents: 4


## 6. Load RAG Pipeline Configs

Load all RAG pipeline configurations from the config directory specified in the experiment config.

In [6]:
# Load RAG pipeline configs
configs = runner.load_configs()

print(f"Loaded {len(configs)} RAG pipeline configurations:")
for cfg in configs:
    print(f"  - {cfg.name}")
    print(f"    Chunking: {cfg.chunking.type.value}")
    print(f"    Retrieval: {cfg.retrieval.type.value}")
    print(f"    Generation: {cfg.generation.config.model}")

Loaded 2 RAG pipeline configurations:
  - high_quality_large
    Chunking: token
    Retrieval: dense_rerank
    Generation: llama-3.1-8b-instant
  - high_quality_small
    Chunking: sentence
    Retrieval: dense_rerank
    Generation: llama-3.1-8b-instant


## 7. Run Experiments

Run all RAG configurations on the loaded data. Each config will:
1. Build a vector index from the documents
2. Run queries against the index
3. Generate responses
4. Evaluate using TRACe metrics

Results are returned as PipelineRunResult objects.

In [7]:
# Run experiments
print(f"Running {len(configs)} configurations on {experiment_config.num_queries} queries...")
print(f"Parallel mode: {experiment_config.parallel}")

runs = runner.run(documents, raw_data)

print(f"\n✅ Experiments completed!")
print(f"  Ran {len(runs)} configurations")

for run in runs:
    print(f"  - {run.config.name}: {len(run.records)} queries")

Running 2 configurations on 2 queries...
Parallel mode: False
Loading embedding model: BAAI/bge-large-en-v1.5


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 8233.29it/s]


Loading reranker model: BAAI/bge-reranker-v2-m3


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 7479.60it/s]



✅ Experiments completed!
  Ran 2 configurations
  - high_quality_large: 2 queries
  - high_quality_small: 2 queries


## 8. Generate Reports

Generate detailed reports for each configuration including:
- Per-query table with all TRACe scores
- Aggregate statistics (mean, std, MAE)
- Comparison with ground truth

In [8]:
# Generate reports
print("Generating reports...")
reports = runner.generate_reports(runs)

print(f"\n✅ Reports generated!")
print(f"  Saved to: {experiment_config.report_dir}")

Generating reports...

✅ Reports generated!
  Saved to: reports


## 9. Display Reports

Display the generated reports with per-query and aggregate metrics.

In [9]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

for report in reports:
    print(f"\n{'='*80}")
    print(f"Configuration: {report.config_name}")
    print(f"{'='*80}")
    
    # Display per-query results
    print("\nPer-Query Results:")
    display(report.display())
    


Configuration: high_quality_large

Per-Query Results:


# RAG Multi-Config Evaluation Report

_Strategy: detailed_query_

## Config: `high_quality_large`

**name**: high_quality_large  •  **mode**: test  •  **providers**: {'groq': {'type': 'groq', 'api_key_env': 'GROQ_API_KEY', 'params': {'cooldown_seconds': 60}}}  •  **chunking**: {'type': 'token', 'config': {'max_tokens': 100, 'overlap_tokens': 10}}  •  **embedding**: {'type': 'sentence_transformer', 'config': {'model_name': 'BAAI/bge-large-en-v1.5', 'model': None, 'dimension': 1024}}  •  **vector_store**: {'type': 'faiss', 'config': {'dimension': 1024}}  •  **retrieval**: {'type': 'dense_rerank', 'config': {'top_k': 5, 'initial_k': 20}}  •  **reranker**: {'type': 'cross_encoder', 'config': {'model_name': 'BAAI/bge-reranker-v2-m3'}}  •  **generation**: {'strategy': 'default', 'provider': 'groq', 'config': {'model': 'llama-3.1-8b-instant', 'max_tokens': 1024, 'temperature': 0.7, 'system_prompt': None}}  •  **evaluation**: {'type': 'trace', 'provider': 'groq', 'config': {'model': 'llama-3.1-8b-instant', 'max_tokens': 2000, 'temperature': 0.0}}  •  **cache**: {'enabled': True, 'cache_dir': 'cache'}

...

### Per-query results

,query,retrieved_documents,answer,relevance_score__pred,relevance_score__gt,relevance_score__deviation,utilization_score__pred,utilization_score__gt,utilization_score__deviation,completeness_score__pred,completeness_score__gt,completeness_score__deviation,adherence_score__pred,adherence_score__gt,adherence_score__deviation
0,Which viruses may not cause prolonged inflammation due to strong induction o...,[{'text': 'successful treatment for HCV serves to circumvent the viral inhib...,"Based on the given context, we can infer that the following viruses may not ...",0.4545,0.4118,0.0427,0.2273,0.1765,0.0508,0.4000,0.4286,-0.0286,0.0,0.0,0.0
1,When was the first case of COVID-19 identified?,"[{'text': 'Abstract: In the WHO European Region, COVID-19 surveillance was i...","Unfortunately, the provided context doesn't explicitly mention the exact dat...",1.0000,0.2692,0.7308,0.1481,0.0769,0.0712,0.1481,0.2857,-0.1376,0.0,1.0,-1.0


### Aggregate TRACe scores (mean / ground truth / deviation)

,metric,mean_score,mean_ground_truth,std_score,std_ground_truth,mean_abs_error
0,relevance_score,0.7272,0.3405,0.2728,0.0713,0.3868
1,utilization_score,0.1877,0.1267,0.0396,0.0498,0.0610
2,completeness_score,0.2740,0.3571,0.1260,0.0714,0.0831
3,adherence_score,0.0000,0.5000,0.0000,0.5000,0.5000


None

## 10. Compare Configurations

Generate a comparison report across all configurations to see which performs best.

In [10]:
# Generate comparison report
print("Generating comparison report...")
comparison = runner.compare()

print(f"\n✅ Comparison report generated!")
print(f"  Saved to: {experiment_config.report_dir}/comparison.csv")

Generating comparison report...

✅ Comparison report generated!
  Saved to: reports/comparison.csv


In [12]:
# Display comparison
print("\nConfiguration Comparison:")
display(comparison.to_dataframe())


Configuration Comparison:


,config_name,metric,mean_score,mean_ground_truth,std_score,std_ground_truth,mean_abs_error
0,high_quality_large,relevance_score,0.4790,0.3405,0.2938,0.0713,0.2225
1,high_quality_small,relevance_score,0.7272,0.3405,0.2728,0.0713,0.3868
2,fast-local,relevance_score,0.8080,0.2345,0.2994,0.1242,0.5955
3,high-quality,relevance_score,0.3669,0.2345,0.2722,0.1242,0.1354


## 11. Summary

The ExperimentRunner provides a complete workflow for:

1. **Configuration-driven data loading** - Specify data source in YAML
2. **Automatic parsing** - Documents parsed using configured parser
3. **Multi-config evaluation** - Test multiple RAG configurations
4. **Parallel execution** - Speed up evaluation with parallel runs
5. **Comprehensive reporting** - Per-query and aggregate metrics
6. **Cross-config comparison** - Identify best performing config

### Key Benefits:

- **Reproducible** - Everything configured in YAML
- **Flexible** - Easy to change data source or parser
- **Scalable** - Parallel execution for faster evaluation
- **Comprehensive** - Detailed metrics and comparisons